# Audio Localization — EDA
**Goal:** Verify the 24-angle dataset is suitable for localization. Check files, signal quality, TDOA patterns, and train/test split.

In [ ]:
import os, wave
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

DATA   = r'C:\Users\ahmma\Desktop\farah\(24 angles)dataset'
ANGLES = list(range(0, 360, 15))
MICS   = ['mic_front', 'mic_back', 'mic_left', 'mic_right']
SR     = 16000

def load(base, angle, dur, mic):
    p = os.path.join(base, str(angle), 'speakerM', dur, mic + '.wav')
    with wave.open(p) as f:
        d = np.frombuffer(f.readframes(f.getnframes()), dtype=np.int16).astype(np.float32)
    return d / 32768.0

def rms_db(s):   return 20 * np.log10(np.sqrt(np.mean(s**2)) + 1e-10)

def gcc_tdoa(s1, s2, max_ms=2):
    n  = len(s1) + len(s2) - 1
    R  = np.fft.rfft(s1, n=n) * np.conj(np.fft.rfft(s2, n=n))
    R /= np.abs(R) + 1e-10
    cc = np.fft.irfft(R, n=n)
    ml = int(SR * max_ms / 1000)
    cc = np.concatenate([cc[-ml:], cc[:ml+1]])
    lags = np.arange(-ml, ml+1) / SR * 1000
    return float(lags[np.argmax(cc)])

print('Setup done.')

## 1. Data Inventory
Check all 192 files exist (24 angles x 4 mics x 2 durations).

In [ ]:
rows = []
for angle in ANGLES:
    for dur in ['5min', '3min']:
        for mic in MICS:
            p = os.path.join(DATA, str(angle), 'speakerM', dur, mic + '.wav')
            if os.path.exists(p):
                with wave.open(p) as f:
                    rows.append({'angle': angle, 'duration': dur, 'mic': mic,
                                 'sr': f.getframerate(),
                                 'dur_s': round(f.getnframes()/f.getframerate(), 1),
                                 'ok': True})
            else:
                rows.append({'angle': angle, 'duration': dur, 'mic': mic, 'ok': False})

inv = pd.DataFrame(rows)
found = inv[inv.ok].shape[0]
missing = inv[~inv['ok']]
print(f"Files found: {found}/192  {'OK' if found==192 else 'MISSING FILES'}")
if len(missing): print(missing[['angle','duration','mic']].to_string(index=False))
print()
print(inv[inv.ok].groupby('duration')[['dur_s','sr']].mean().round(1))

## 2. Signal Level per Angle
RMS across all 24 angles for mic_front. Should be stable — large variation means recording issues.

In [ ]:
stats = []
for angle in ANGLES:
    s = load(DATA, angle, '5min', 'mic_front')
    stats.append({'angle': angle, 'rms_db': round(rms_db(s), 2),
                  'clipping': bool(np.any(np.abs(s) >= 1.0))})

df_stats = pd.DataFrame(stats)
print(f"Mean RMS: {df_stats.rms_db.mean():.1f} dBFS  |  Std: {df_stats.rms_db.std():.2f} dB")
print(f"Clipping: {df_stats['clipping'].sum()} files")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df_stats.angle, df_stats.rms_db, marker='o', linewidth=1.5, color='steelblue')
ax.set(xlabel='Angle (deg)', ylabel='RMS (dBFS)', title='RMS per Angle — mic_front 5min')
ax.set_xticks(ANGLES); ax.tick_params(axis='x', rotation=90)
ax.grid(True, alpha=0.4); plt.tight_layout(); plt.show()

## 3. TDOA per Angle
GCC-PHAT time-delay between mic pairs. Each angle must produce a distinct TDOA pattern for localization to work.

In [ ]:
pairs = [
    ('mic_front', 'mic_back',  'FB'),
    ('mic_left',  'mic_right', 'LR'),
    ('mic_front', 'mic_left',  'FL'),
    ('mic_front', 'mic_right', 'FR'),
]

tdoa_rows = []
for angle in ANGLES:
    row = {'angle': angle}
    for m1, m2, label in pairs:
        s1 = load(DATA, angle, '5min', m1)
        s2 = load(DATA, angle, '5min', m2)
        row[label] = gcc_tdoa(s1, s2)
    tdoa_rows.append(row)

df_tdoa = pd.DataFrame(tdoa_rows)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, (_, _, label) in zip(axes.flatten(), pairs):
    ax.plot(df_tdoa.angle, df_tdoa[label], marker='o', linewidth=1.5)
    ax.axhline(0, color='gray', linewidth=0.8, linestyle=':')
    ax.set(xlabel='Angle (deg)', ylabel='TDOA (ms)', title=f'TDOA {label}')
    ax.set_xticks(ANGLES); ax.tick_params(axis='x', rotation=90)
    ax.grid(True, alpha=0.4)
plt.suptitle('TDOA per Angle — 24 angles', fontsize=12)
plt.tight_layout(); plt.show()

## 4. Feature Separability
Fisher ratio per feature — higher means the feature distinguishes angles better.

In [ ]:
from scipy.signal import hilbert

CHUNK    = 2 * SR
features = {k: {a: [] for a in ANGLES} for k in ['RMS','TDOA_FB','TDOA_LR','IPD_FB']}

for angle in ANGLES:
    sigs = {m: load(DATA, angle, '5min', m) for m in MICS}
    n_chunks = min(len(s) for s in sigs.values()) // CHUNK
    for ci in range(n_chunks):
        sl = slice(ci*CHUNK, (ci+1)*CHUNK)
        ch = {m: sigs[m][sl] for m in MICS}
        features['RMS'][angle].append(rms_db(ch['mic_front']))
        features['TDOA_FB'][angle].append(gcc_tdoa(ch['mic_front'], ch['mic_back']))
        features['TDOA_LR'][angle].append(gcc_tdoa(ch['mic_left'],  ch['mic_right']))
        a1 = np.angle(hilbert(ch['mic_front']))
        a2 = np.angle(hilbert(ch['mic_back']))
        features['IPD_FB'][angle].append(float(np.mean(np.degrees(np.arctan2(np.sin(a1-a2), np.cos(a1-a2))))))

def fisher_ratio(values_per_class):
    all_vals = np.concatenate(values_per_class)
    overall_mean = np.mean(all_vals)
    between = np.mean([(np.mean(v) - overall_mean)**2 for v in values_per_class])
    within  = np.mean([np.var(v) for v in values_per_class])
    return between / (within + 1e-10)

results = {name: fisher_ratio(list(d.values())) for name, d in features.items()}
results_sorted = sorted(results.items(), key=lambda x: x[1], reverse=True)

print("Fisher Ratio (higher = better):")
for name, score in results_sorted:
    print(f"  {name:<12} {score:.2f}")

fig, ax = plt.subplots(figsize=(7, 4))
names, scores = zip(*results_sorted)
colors = ['#1D9E75' if s > 5 else '#378ADD' if s > 2 else '#E24B4A' for s in scores]
ax.barh(names, scores, color=colors, edgecolor='black', linewidth=1.2)
ax.axvline(2, color='gray', linestyle='--', linewidth=1, label='Moderate (>2)')
ax.axvline(5, color='green', linestyle='--', linewidth=1, label='Good (>5)')
ax.set(xlabel="Fisher Ratio", title='Feature Separability — 24 angles')
ax.legend(); ax.grid(True, axis='x', alpha=0.4)
plt.tight_layout(); plt.show()

## 5. Train / Test Split Validation
5min = train, 3min = test. RMS difference confirms they are different recordings.

In [ ]:
split_rows = []
for angle in ANGLES:
    s5 = load(DATA, angle, '5min', 'mic_front')
    s3 = load(DATA, angle, '3min', 'mic_front')
    split_rows.append({'angle': angle,
                       'rms_5min': round(rms_db(s5), 2),
                       'rms_3min': round(rms_db(s3), 2),
                       'rms_diff': round(abs(rms_db(s5) - rms_db(s3)), 2)})

df_split = pd.DataFrame(split_rows)
print(df_split[['angle','rms_5min','rms_3min','rms_diff']].to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(df_split.angle, df_split.rms_diff, color='steelblue', width=10, alpha=0.8)
ax.set(xlabel='Angle (deg)', ylabel='|RMS diff| (dB)', title='|5min - 3min| RMS diff per Angle')
ax.set_xticks(ANGLES); ax.tick_params(axis='x', rotation=90)
ax.grid(True, axis='y', alpha=0.4); plt.tight_layout(); plt.show()

## Summary

In [ ]:
print("=" * 50)
print("  DATASET READINESS CHECKLIST")
print("=" * 50)
total = inv[inv.ok].shape[0]
print(f"  Files found    : {total}/192  {'OK' if total==192 else 'MISSING'}")
print(f"  Clipping       : {df_stats['clipping'].sum()} files  {'OK' if df_stats['clipping'].sum()==0 else 'CHECK'}")
print(f"  Mean RMS       : {df_stats.rms_db.mean():.1f} dBFS")
best_feat, best_score = results_sorted[0]
print(f"  Best feature   : {best_feat}  (Fisher={best_score:.2f})")
min_diff = df_split.rms_diff.min()
print(f"  Train/test sep : min RMS diff={min_diff:.2f} dB  {'OK' if min_diff > 0 else 'WARNING'}")
print()
print("  Train : 5min recordings")
print("  Test  : 3min recordings")
print("  Model : 24-class CNN")
print("=" * 50)